# nanoGPT With Parallelism, From First Principles

[nanoGPT](https://github.com/karpathy/nanoGPT) made a GPT training loop feel inspectable: one model file, one training file, and very little ceremony. This series tries to do the same thing for distributed transformer parallelism.

We will start with nanoGPT as the reference implementation and add the major parallelism axes one at a time:

```text
tensor parallelism   -> split big linear layers
pipeline parallelism -> split layers across ranks
sequence parallelism -> split token activations
context parallelism  -> split long-context attention state
expert parallelism   -> route tokens to expert MLPs
sharded checkpoints  -> save what each rank owns
```

Terminology note: NVIDIA commonly uses "5D parallelism" for TP, PP, DP, CP, and EP. In this series, DP is treated as the familiar outer replication axis, while the hands-on focus is TP, PP, SP, CP, EP, and sharded checkpointing inside a nanoGPT-shaped model.

Related NVIDIA references:

- [Megatron Core Parallelism Strategies Guide](https://docs.nvidia.com/megatron-core/developer-guide/latest/user-guide/parallelism-guide.html)
- [NeMo AutoModel distributed-training config table](https://docs.nvidia.com/nemo/automodel/recipes-e2e-examples/sft-peft)
- [NeMo Context Parallelism](https://docs.nvidia.com/nemo-framework/user-guide/25.02/longcontext/contextparallel.html)

By the end, the goal is to converge to a small but runnable parallel version of nanoGPT. Not a production trainer, and not an official nanoGPT variant, but a readable implementation where each parallel piece can be traced back to the single-process original.

First-principles means we start from the original tensor equation, use tiny tensors, slice weights by hand, compute each rank's local values, show where communication is required, and only then package the idea into reusable modules.

This notebook is the blog-first version of the repo implementation. We will not start with `ColumnParallelLinear` or `RowParallelLinear`. We will start with nanoGPT's plain MLP, slice its weights by hand, and only then package the idea.

**Note:** I am building this series with Codex as a coding partner to scaffold experiments, run checks, and keep the repo, notebooks, and posts synchronized. The series is about distributed transformer parallelism; Codex is part of the workflow.

**Launch this notebook:** these links will work after the fork is pushed to a public GitHub repo. Replace `OWNER/REPO` with the published repository path.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/01_tp_mlp_from_first_principles.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/notebooks/welcome?src=https://github.com/OWNER/REPO/blob/main/notebooks/01_tp_mlp_from_first_principles.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/OWNER/REPO/main?labpath=notebooks/01_tp_mlp_from_first_principles.ipynb)

## 01. Tensor-Parallel MLP From First Principles

**Audience:** readers who know basic PyTorch and have seen a transformer block.

**Goal:** by the end, you should be able to explain why Megatron-style MLP tensor parallelism uses:

```text
c_fc:   column parallel
GELU:   local
c_proj: row parallel + all_reduce
```


## Outline

1. Load nanoGPT's original MLP.
2. Run one tiny input through the unsharded MLP.
3. Split `c_fc` across two fake tensor-parallel ranks.
4. Apply GELU locally.
5. Split `c_proj` across the hidden dimension.
6. Sum rank-local partial outputs and compare with the original MLP.
7. Build a map of row-parallel, column-parallel, gather, and all-reduce decisions.
8. Estimate the memory/compute benefit of TP.
9. Look at a case where TP/SP can be net negative: LoRA/adapters.
10. Hand off to the clean repo implementation.

In [ ]:
from pathlib import Path
import inspect
import math
import sys

import torch
from torch.nn import functional as F

# Run this notebook from the repo root:
#   cd nanogpt_parallel
#   jupyter notebook notebooks/01_tp_mlp_from_first_principles.ipynb
ROOT = Path.cwd()
assert (ROOT / "model.py").exists(), f"Run from repo root, got {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model import GPTConfig, MLP

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. The Original nanoGPT MLP

nanoGPT's MLP is deliberately simple: expand the channel dimension, apply GELU, project back down.

In [ ]:
print(inspect.getsource(MLP))


## 2. A Tiny Reference Run

We'll use `n_embd = 4`, so the hidden MLP dimension is `4 * n_embd = 16`. That makes two-rank tensor parallelism easy to see: each rank owns 8 hidden features.

In [ ]:
config = GPTConfig(
    block_size=8,
    vocab_size=32,
    n_layer=1,
    n_head=2,
    n_embd=4,
    dropout=0.0,
    bias=True,
)

mlp = MLP(config)
x = torch.tensor([[[-1.0, 0.0, 1.0, 2.0],
                   [ 2.0, 1.0, 0.0,-1.0]]])

with torch.no_grad():
    y_ref = mlp(x)

print("x", x.shape)
print(x)
print("\ny_ref", y_ref.shape)
print(y_ref)


## 3. Look At The Weight Shapes

PyTorch stores linear weights as `[out_features, in_features]`.

```text
c_fc.weight:   [16, 4]
c_proj.weight: [4, 16]
```

The first matrix is split along output features. The second is split along input features.

In [ ]:
print("c_fc.weight", tuple(mlp.c_fc.weight.shape))
print("c_fc.bias  ", tuple(mlp.c_fc.bias.shape))
print("c_proj.weight", tuple(mlp.c_proj.weight.shape))
print("c_proj.bias  ", tuple(mlp.c_proj.bias.shape))


## 4. Column-Parallel `c_fc`

`c_fc` maps from `n_embd` to `4 * n_embd`.

In tensor parallelism, rank 0 gets the first half of output features and rank 1 gets the second half. Both ranks receive the same input `x`, but produce different hidden columns.

In [ ]:
tp = 2
hidden = 4 * config.n_embd
hidden_per_rank = hidden // tp

fc_rank0_w = mlp.c_fc.weight[:hidden_per_rank, :]
fc_rank0_b = mlp.c_fc.bias[:hidden_per_rank]
fc_rank1_w = mlp.c_fc.weight[hidden_per_rank:, :]
fc_rank1_b = mlp.c_fc.bias[hidden_per_rank:]

h0_pre = F.linear(x, fc_rank0_w, fc_rank0_b)
h1_pre = F.linear(x, fc_rank1_w, fc_rank1_b)

print("rank 0 pre-GELU hidden", h0_pre.shape)
print(h0_pre)
print("\nrank 1 pre-GELU hidden", h1_pre.shape)
print(h1_pre)


## 5. GELU Is Local

GELU is elementwise. It does not mix hidden features, so each rank can apply it to its own slice without communication.

In [ ]:
h0 = F.gelu(h0_pre)
h1 = F.gelu(h1_pre)
h_full_reconstructed = torch.cat([h0, h1], dim=-1)

with torch.no_grad():
    h_full_direct = F.gelu(mlp.c_fc(x))

print("rank 0 post-GELU", h0.shape)
print(h0)
print("\nrank 1 post-GELU", h1.shape)
print(h1)
print("\nmax diff vs full hidden:", (h_full_reconstructed - h_full_direct).abs().max().item())


## 6. Row-Parallel `c_proj`

`c_proj` maps from `4 * n_embd` back to `n_embd`.

Now each rank owns part of the input hidden dimension. Each rank computes a partial output of shape `[B, T, n_embd]`. The true output is the sum of those partial outputs plus one copy of the bias.

In [ ]:
proj_rank0_w = mlp.c_proj.weight[:, :hidden_per_rank]
proj_rank1_w = mlp.c_proj.weight[:, hidden_per_rank:]

y0_partial = F.linear(h0, proj_rank0_w, bias=None)
y1_partial = F.linear(h1, proj_rank1_w, bias=None)
y_tp_manual = y0_partial + y1_partial + mlp.c_proj.bias

print("rank 0 partial output", y0_partial.shape)
print(y0_partial)
print("\nrank 1 partial output", y1_partial.shape)
print(y1_partial)
print("\nmanual TP output")
print(y_tp_manual)
print("\nreference output")
print(y_ref)
print("\nmax diff:", (y_tp_manual - y_ref).abs().max().item())


## 7. Where `all_reduce` Appears

In the manual cell above, this line:

```python
y_tp_manual = y0_partial + y1_partial + mlp.c_proj.bias
```

is the single-process version of a distributed `all_reduce(SUM)`.

```text
rank 0 has y0_partial
rank 1 has y1_partial
all_reduce SUM gives both ranks y0_partial + y1_partial
then each rank adds one copy of c_proj.bias
```

In [ ]:
print("The row-parallel communication payload has shape:", tuple(y0_partial.shape))
print("That is B*T*n_embd elements:", y0_partial.numel())


## 8. A Small Parallelism Map

Once you have seen column-parallel `c_fc` and row-parallel `c_proj`, it helps to place them in a wider transformer map.

The names are slightly confusing because PyTorch stores linear weights as `[out_features, in_features]`, while the parallelism names come from the mathematical matrix layout.

| Transformer piece | Common sharding | Parallel type | Communication after op? | Why |
| --- | --- | --- | --- | --- |
| MLP `c_fc` / up projection | output hidden features | Column parallel | No gather before GELU | Each rank owns different expanded hidden features; GELU is elementwise. |
| MLP GELU / activation | local hidden shard | Elementwise local | None | No feature mixing. |
| MLP `c_proj` / down projection | input hidden features | Row parallel | `all_reduce(SUM)` | Each rank computes a partial residual-stream output. |
| Attention `c_attn` QKV | attention heads / QKV output features | Column parallel | Usually no gather before attention | Each rank can run local heads. |
| Attention scores + softmax + value mix | local heads | Head parallel | None for TP | Heads are independent until the output projection. |
| Attention `c_proj` | head/output input features | Row parallel | `all_reduce(SUM)` | Local heads produce partial residual-stream outputs. |
| Residual add after TP projection | replicated `n_embd` stream | Replicated | None after prior all-reduce | The previous row-parallel projection already reconstructed the stream. |
| LayerNorm in plain TP | replicated `n_embd` stream | Replicated | None | LN sees the full hidden dimension if sequence parallelism is not active. |
| LayerNorm with sequence parallelism | sequence tokens | Sequence parallel | `all_gather` or `reduce_scatter` around boundary ops | Saves activation memory, but introduces extra movement. |
| LM head / logits | vocabulary dimension | Vocab parallel | Avoid full gather if using vocab-parallel loss | Gathering logits can be huge; better to reduce loss statistics. |
| LoRA / adapters | framework-specific | Often replicated for SP, TP may mirror the base layer | Depends on target layer | Adapter rank is small, so frameworks try to avoid communication that does not buy enough memory or compute savings. |

In [ ]:
parallelism_map = [
    ("MLP c_fc", "column parallel", "no gather before GELU"),
    ("MLP c_proj", "row parallel", "all_reduce SUM"),
    ("Attention QKV", "column parallel over heads", "no gather before attention"),
    ("Attention c_proj", "row parallel over heads", "all_reduce SUM"),
    ("Plain LayerNorm", "replicated", "none"),
    ("Sequence-parallel boundary", "sequence tokens", "all_gather or reduce_scatter"),
    ("LoRA/adapters", "framework-specific", "replicate for SP or mirror base TP"),
]

for name, kind, comm in parallelism_map:
    print(f"{name:28s} | {kind:32s} | {comm}")


The pattern:

```text
column parallel  -> produces sharded features -> no gather if the next op is local
row parallel     -> consumes sharded features -> all_reduce to rebuild full output
sequence parallel -> saves activation memory -> needs gathers/reduce-scatters at boundaries
```

Good implementations compose layers so that one layer's sharded output is exactly the next layer's expected sharded input.

## 9. What Benefit Do We Get?

For one nanoGPT MLP block, ignoring dropout:

```text
c_fc params   = n_embd * 4*n_embd + 4*n_embd
c_proj params = 4*n_embd * n_embd + n_embd
total         = 8*n_embd^2 + 5*n_embd
```

With tensor parallel size `p`, the large matrix weights and the expanded hidden activation are divided roughly by `p` per rank. The tradeoff is communication: row-parallel `c_proj` needs an `all_reduce` over `[B, T, n_embd]`.

In [ ]:
def mlp_tp_table(n_embd=768, batch=12, seq=1024, dtype_bytes=2, training_state_bytes_per_param=16):
    rows = []
    full_params = 8 * n_embd * n_embd + 5 * n_embd
    full_hidden = batch * seq * 4 * n_embd
    for p in [1, 2, 4, 8]:
        per_rank_params = (8 * n_embd * n_embd) / p + (4 * n_embd) / p + n_embd
        per_rank_hidden = full_hidden / p
        all_reduce_elems = batch * seq * n_embd
        rows.append({
            "tp": p,
            "mlp_params_per_rank_M": per_rank_params / 1e6,
            "param_state_per_rank_MB": per_rank_params * training_state_bytes_per_param / 1e6,
            "hidden_activation_per_rank_MB": per_rank_hidden * dtype_bytes / 1e6,
            "row_parallel_all_reduce_MB": all_reduce_elems * dtype_bytes / 1e6,
        })
    return rows

rows = mlp_tp_table()
print("Assume n_embd=768, batch=12, seq=1024, bf16 activations, rough 16 bytes/param training state.")
print(f"{'TP':>2} | {'params/rank M':>13} | {'param state/rank MB':>19} | {'hidden act/rank MB':>19} | {'all_reduce MB':>13}")
print("-" * 82)
for r in rows:
    print(
        f"{r['tp']:>2} | "
        f"{r['mlp_params_per_rank_M']:>13.2f} | "
        f"{r['param_state_per_rank_MB']:>19.1f} | "
        f"{r['hidden_activation_per_rank_MB']:>19.1f} | "
        f"{r['row_parallel_all_reduce_MB']:>13.1f}"
    )


**Interpretation.** Per rank, TP reduces MLP parameter state and the expanded hidden activation almost linearly. The all-reduce payload does not shrink with TP in this simple layout; it is the cost we pay to reconstruct the projected residual stream.

## 10. When TP/SP Is Not Worth It

The table also hints at an important negative result: not every tensor should be parallelized.

LoRA is a good example. A LoRA update for one linear layer is:

```text
delta(x) = x @ A @ B
rank r << n_embd
```

The trainable matrices are tiny compared with the frozen base matrix. Sharding them may save only tens or hundreds of kilobytes per rank, while adding gathers, reduce-scatters, or all-reduces in the forward and backward path. This is especially easy to miss when LoRA is attached to a TP'd base linear. The dense base path may use a column-parallel projection followed by a row-parallel projection because the expanded hidden activation is large. The LoRA path is much smaller, so production implementations treat it more selectively. For example, NeMo AutoModel has a SequenceParallelLora style that replicates LoRA parameters under sequence parallelism. Megatron Bridge is more nuanced: its adapter mirrors the target base layer. If LoRA is attached to a row-parallel base layer such as linear_fc2, the adapter can still use a row-parallel LoRA-A projection and then a column-parallel LoRA-B projection. It also has special sequence-parallel gather/scatter paths and disables some of those paths when the adapter or target layer is not eligible. So the practical lesson is not "LoRA never uses TP" or "LoRA always skips row parallelism." The lesson is that LoRA is small enough that frameworks make case-by-case choices: replicate under SP in some paths, mirror the base TP layout in others, and avoid extra communication when it is unlikely to pay for it.

In [ ]:
def lora_vs_full(n_embd=4096, rank=8, dtype_bytes=2):
    full_params = n_embd * n_embd
    lora_params = n_embd * rank + rank * n_embd
    return {
        "full_weight_params_M": full_params / 1e6,
        "lora_params_K": lora_params / 1e3,
        "lora_as_percent_of_full": 100 * lora_params / full_params,
        "lora_weight_MB": lora_params * dtype_bytes / 1e6,
    }

stats = lora_vs_full()
for k, v in stats.items():
    print(f"{k:28s}: {v:.4f}")


So the rule is not "TP everything." The rule is:

```text
Shard tensors whose memory/compute savings dominate the communication.
Replicate tensors whose size is small enough that communication would dominate.
```

## 11. Exercise

Change `n_embd`, `batch`, and `seq` in the table above. Try:

```text
n_embd=4096, batch=4, seq=2048
```

What grows faster: parameter state or activation memory?

In [ ]:
# Answer scaffold: uncomment and run.
# rows = mlp_tp_table(n_embd=4096, batch=4, seq=2048)
# for r in rows:
#     print(r)


## 12. The Clean Repo Implementation

The repo packages the derivation into:

```text
parallel/linear.py  -> ColumnParallelLinear, RowParallelLinear
parallel/mlp.py     -> TensorParallelMLP
labs/02_tp_mlp.py   -> 2-rank trace and correctness check
```

Run the distributed version from the repo root:

In [ ]:
!python -m torch.distributed.run --standalone --nproc_per_node=2 labs/02_tp_mlp.py --trace

## Pitfall

Do not add the `c_proj` bias before the `all_reduce`. If each rank adds the full bias to its partial output and then the partials are summed, the bias is counted once per rank.

Correct:

```text
all_reduce(sum partial outputs)
add bias once
```